# Contextual Compression Retriever — 맥락 기반 문서 압축 검색기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **ContextualCompressionRetriever**의 작동 원리 이해하기
- `LLMChainExtractor`, `LLMChainFilter`, `EmbeddingsFilter` 각각의 차이 파악하기
- `DocumentCompressorPipeline`으로 여러 압축 단계를 조합하기

## 사전 지식

- 01-VectorStoreRetriever에서 기본 Retriever를 만들어본 경험
- LLM(대규모 언어 모델)과 임베딩 모델의 역할 이해

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> BR[Base Retriever<br/>후보 문서 검색]:::process
    BR --> C{Compressor<br/>선택}:::process
    C -- LLMExtractor --> E1[LLM이<br/>관련 내용 추출]:::process
    C -- LLMFilter --> E2[LLM이<br/>문서 선별]:::process
    C -- EmbeddingsFilter --> E3[임베딩<br/>유사도 필터]:::process
    E1 --> O[압축된<br/>문서]:::output
    E2 --> O
    E3 --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 문제

일반 Retriever는 문서 전체를 반환하지만, 실제로 필요한 정보는 문서의 일부분일 수 있어요:

- **문제 1**: 무관한 텍스트가 많이 포함되어 LLM 비용 증가
- **문제 2**: 노이즈가 많아 응답 품질 저하
- **문제 3**: 컨텍스트 길이 제한으로 중요 정보 누락

## 해결 방법

`ContextualCompressionRetriever`는 두 단계로 작동해요:

1. **검색(Retrieval)**: Base Retriever로 후보 문서 검색
2. **압축(Compression)**: Document Compressor로 관련 내용만 추출

> 🔑 **핵심 개념**: Contextual Compression은 검색 품질과 LLM 비용을 동시에 개선합니다. 불필요한 텍스트를 제거하면 토큰 수가 줄어들고, 남은 내용은 쿼리와 더 관련성이 높아져요.

## 환경 설정


In [1]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import (
    DocumentCompressorPipeline,
    EmbeddingsFilter,
    LLMChainExtractor,
    LLMChainFilter,
)


## 1. 기본 Retriever 준비

먼저 압축하지 않은 기본 retriever의 결과를 확인해봅시다.


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# 압축 기법 비교를 위한 기본 Retriever 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 800자 청크로 분할한 뒤 FAISS Retriever를 생성하세요
# 힌트: chunk_size=800으로 크게 설정해야 압축 효과가 잘 보임
# 예상 결과: 총 청크 수와 기본 Retriever 생성 완료 메시지 출력
# ============================================================

# 문서 로드 및 분할
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

# 큰 청크를 사용해야 압축 후 크기 감소 효과가 뚜렷이 보임
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # 비교를 위해 큰 청크 사용
    chunk_overlap=100
)
split_docs = text_splitter.split_documents(documents)

# 벡터 데이터베이스 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)

# 기본 retriever 생성
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ 기본 Retriever 준비 완료")
print(f"총 청크 수: {len(split_docs)}")
print(f"청크 크기: 800자")
print(f"검색할 문서 수: 3개")

✅ 기본 Retriever 준비 완료
총 청크 수: 11
청크 크기: 800자
검색할 문서 수: 3개


In [ ]:
# 헬퍼 함수: 문서를 깔끔하게 출력
def print_docs(docs, title="검색 결과"):
    print(f"<br/>{'='*80}")
    print(f"{title}")
    print(f"{'='*80}<br/>")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(f"내용: {doc.page_content}")
        print(f"길이: {len(doc.page_content)} 문자")
        print("-" * 80 + "<br/>")


In [ ]:
# ---------------------------------------------------
# 기본 Retriever 결과를 확인해 압축 전 상태 파악
# ---------------------------------------------------

# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 총 문자 수를 계산하세요
# 힌트: retriever.invoke(query) 후 sum(len(d.page_content) for d in docs)
# 예상 결과: 원본 크기 확인 (압축 후와 비교할 기준선)
# ============================================================

# 테스트 쿼리
query = "딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?"

# 기본 retriever로 검색
docs = base_retriever.invoke(query)

print(f"📝 쿼리: {query}<br/>")
print_docs(docs, "기본 Retriever 결과 (압축 없음)")

# 전체 문자 수 계산
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"<br/>💡 총 문자 수: {total_chars}자")
print("⚠️ 문제: 쿼리와 무관한 내용도 많이 포함되어 있음")

📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?


기본 Retriever 결과 (압축 없음)

[문서 1]
내용: 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는 데 매우 효과적이며, 문장의 길이에 관계없이 일정한 연산 시간으로 처리할 수 있습니다.

"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으로써 발생하는 병렬 처리의 이점을 강조하며, 이를 통해 훨씬 더 빠른 학습 속도와 효율적인 메모리 사용을 실현합니다.

변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔습니다. BERT, GPT, RoBERTa 등의 후속 모델들은 모두 "Attention Is All You Need" 논문에서 제시된 변환기 아키텍처를 기반으로 하며, 이들 모델은 텍스트 분류, 요약, 번역, 질문 응답 시스템 등 다양한 작업에서 인상적인 성능을 보여줍니다.
길이: 721 문자
--------------------------------------------------------------------------------

[문서 2]
내용: Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영

## 2. LLMChainExtractor - LLM 기반 내용 추출

`LLMChainExtractor`는 LLM을 사용하여 쿼리와 관련된 내용만 추출합니다.

### 작동 방식

1. 검색된 각 문서에 대해 LLM에게 질문
2. "이 문서에서 쿼리와 관련된 부분만 추출하세요"
3. LLM이 관련 내용만 반환

### 장점과 단점

**장점:**
- 매우 정확한 내용 추출
- 문맥을 이해하여 관련성 판단

**단점:**
- LLM 호출 비용 발생
- 속도가 느림 (각 문서마다 LLM 호출)

> 🎯 **강의 포인트**: `LLMChainExtractor`는 가장 정확하지만 가장 비쌉니다. 문서가 3개라면 LLM을 3번 추가 호출해요. 품질이 최우선인 내부 도구나 소량 문서 처리에 적합합니다.

> ⚠️ **자주 하는 실수**: LLMChainExtractor를 모든 상황에 쓰려는 경향이 있어요. 실시간 응답이 필요하거나 문서가 많다면 EmbeddingsFilter가 훨씬 경제적입니다.

In [6]:
from langchain_openai import ChatOpenAI

# LLM 초기화
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# LLMChainExtractor 생성
compressor = LLMChainExtractor.from_llm(llm)

# ContextualCompressionRetriever 생성
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("✅ LLMChainExtractor 기반 Compression Retriever 생성 완료")


✅ LLMChainExtractor 기반 Compression Retriever 생성 완료


In [ ]:
# 압축된 검색 실행
compressed_docs = compression_retriever.invoke(query)

print(f"📝 쿼리: {query}<br/>")
print_docs(compressed_docs, "LLMChainExtractor 적용 후")

# 압축 효과 비교
original_total = sum(len(doc.page_content) for doc in docs)
compressed_total = sum(len(doc.page_content) for doc in compressed_docs)
reduction = (1 - compressed_total / original_total) * 100

print(f"<br/>📊 압축 효과:")
print(f"  원본: {original_total}자")
print(f"  압축 후: {compressed_total}자")
print(f"  감소율: {reduction:.1f}%")
print(f"<br/>💡 장점: 쿼리와 직접 관련된 내용만 추출되어 품질 향상")


📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?


LLMChainExtractor 적용 후

[문서 1]
내용: 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는 데 매우 효과적이며, 문장의 길이에 관계없이 일정한 연산 시간으로 처리할 수 있습니다. 

"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다.
길이: 419 문자
--------------------------------------------------------------------------------

[문서 2]
내용: "Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Transformer) 모델의 아키텍처를 처음으로 소개한 연구입니다. Ashish Vaswani와 그의 공동 연구자들에 의해 작성된 이 논문은 자연어 처리(NLP)와 기계 학습 분야에 큰 영향을 미쳤습니다. 변환기 모델은 이전의 순환 신경망(RNN)이나 합성곱 신경망(CNN)에 의존하던 방식에서 벗어나, 전적으로 '어텐션 메커니즘(Attention Mechanism)'을 사용하여 정보를 처리합니다. 이로 인해 모델의 학습 속도와 효율성이 크게 향상되었습니다.
길이: 308 문자
--------------------------------------------------------------------------------

## 3. LLMChainFilter - LLM 기반 문서 필터링

`LLMChainFilter`는 문서 내용을 압축하지 않고, 관련 있는 문서만 선택적으로 반환합니다.

### LLMChainExtractor vs LLMChainFilter

| 특징 | LLMChainExtractor | LLMChainFilter |
|------|------------------|----------------|
| **동작** | 문서 내용 추출/압축 | 문서 전체 유지 또는 제거 |
| **결과** | 압축된 내용 반환 | 원본 문서 또는 빈 결과 |
| **용도** | 긴 문서에서 핵심 추출 | 관련 없는 문서 제거 |
| **속도** | 느림 | 느림 |
| **비용** | 높음 (추출 작업) | 높음 (판단 작업) |


In [ ]:
# ---------------------------------------------------
# LLMChainFilter로 관련 없는 문서 제거 (내용 압축 없이)
# ---------------------------------------------------

# ============================================================
# TODO: LLMChainFilter를 사용해 관련 없는 문서를 제거하는 retriever를 만들고 검색하세요
# 힌트: LLMChainFilter.from_llm(llm) → ContextualCompressionRetriever
# 예상 결과: 원본 문서를 그대로 유지하되 관련 없는 문서는 제거됨
# ============================================================

# LLMChainFilter 생성
# LLMChainFilter: 문서 내용을 압축하지 않고 관련/비관련 이진 판단
doc_filter = LLMChainFilter.from_llm(llm)

# ContextualCompressionRetriever 생성
filter_retriever = ContextualCompressionRetriever(
    base_compressor=doc_filter,
    base_retriever=base_retriever
)

# 필터링된 검색 실행
filtered_docs = filter_retriever.invoke(query)

print(f"📝 쿼리: {query}<br/>")
print_docs(filtered_docs, "LLMChainFilter 적용 후")

print(f"<br/>💡 특징:")
print(f"  - 원본 문서: {len(docs)}개")
print(f"  - 필터링 후: {len(filtered_docs)}개")
print(f"  - 관련 없는 문서는 완전히 제거됨")
print(f"  - 남은 문서는 원본 그대로 유지")

📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?


LLMChainFilter 적용 후

[문서 1]
내용: 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는 데 매우 효과적이며, 문장의 길이에 관계없이 일정한 연산 시간으로 처리할 수 있습니다.

"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으로써 발생하는 병렬 처리의 이점을 강조하며, 이를 통해 훨씬 더 빠른 학습 속도와 효율적인 메모리 사용을 실현합니다.

변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔습니다. BERT, GPT, RoBERTa 등의 후속 모델들은 모두 "Attention Is All You Need" 논문에서 제시된 변환기 아키텍처를 기반으로 하며, 이들 모델은 텍스트 분류, 요약, 번역, 질문 응답 시스템 등 다양한 작업에서 인상적인 성능을 보여줍니다.
길이: 721 문자
--------------------------------------------------------------------------------

[문서 2]
내용: Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 대

## 4. EmbeddingsFilter - 임베딩 기반 필터링

`EmbeddingsFilter`는 임베딩 유사도를 사용하여 빠르고 저렴하게 문서를 필터링합니다.

### 장점

- **빠른 속도**: LLM 호출 없이 벡터 연산만 수행
- **낮은 비용**: 임베딩 유사도 계산만 필요
- **효율성**: 대량의 문서 필터링에 적합

### 언제 사용하면 좋을까요?

- 빠른 응답이 필요한 애플리케이션
- 비용을 최소화해야 할 때
- 이미 검색된 문서를 추가로 필터링할 때

> 💡 **실무 팁**: 프로덕션 환경에서 Compression Retriever가 필요하다면 EmbeddingsFilter를 먼저 선택하세요. LLM 호출 없이 동작하므로 레이턴시 증가가 거의 없고, 비용도 절감됩니다.

> 🔑 **핵심 개념**: EmbeddingsFilter의 `similarity_threshold`는 쿼리와 문서 임베딩 간의 코사인 유사도 기준입니다. 0.75가 일반적인 시작점이지만, 도메인에 따라 조정이 필요해요.

In [ ]:
# EmbeddingsFilter 생성 (유사도 임계값 0.75)
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.472
)

# ContextualCompressionRetriever 생성
embedding_filter_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

# 임베딩 기반 필터링 실행
embedding_filtered_docs = embedding_filter_retriever.invoke(query)

print(f"📝 쿼리: {query}<br/>")
print_docs(embedding_filtered_docs, "EmbeddingsFilter 적용 후")

print(f"<br/>💡 특징:")
print(f"  - 유사도 임계값: 0.472")
print(f"  - LLM 호출 없음 (빠르고 저렴)")
print(f"  - 원본 문서: {len(docs)}개 → 필터링 후: {len(embedding_filtered_docs)}개")


📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?


EmbeddingsFilter 적용 후

[문서 1]
내용: 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는 데 매우 효과적이며, 문장의 길이에 관계없이 일정한 연산 시간으로 처리할 수 있습니다.

"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으로써 발생하는 병렬 처리의 이점을 강조하며, 이를 통해 훨씬 더 빠른 학습 속도와 효율적인 메모리 사용을 실현합니다.

변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔습니다. BERT, GPT, RoBERTa 등의 후속 모델들은 모두 "Attention Is All You Need" 논문에서 제시된 변환기 아키텍처를 기반으로 하며, 이들 모델은 텍스트 분류, 요약, 번역, 질문 응답 시스템 등 다양한 작업에서 인상적인 성능을 보여줍니다.
길이: 721 문자
--------------------------------------------------------------------------------


💡 특징:
  - 유사도 임계값: 0.472
  - LLM 호출 없음 (빠르고 저렴)
  - 원본 문서: 3개 → 필터링 후: 1개


## 5. DocumentCompressorPipeline - 여러 Compressor 조합

`DocumentCompressorPipeline`을 사용하면 여러 compressor와 transformer를 순차적으로 적용할 수 있습니다.

### 파이프라인 구성 요소

1. **TextSplitter**: 문서를 더 작은 청크로 분할
2. **EmbeddingsRedundantFilter**: 중복 문서 제거 (유사도 0.95 이상)
3. **EmbeddingsFilter**: 관련성 필터링 (유사도 임계값)
4. **LLMChainExtractor**: 최종 내용 추출

### 파이프라인 전략

```
큰 문서 → 작은 청크 → 중복 제거 → 관련성 필터링 → 내용 추출
```

> 🎯 **강의 포인트**: Pipeline은 각 단계가 다음 단계의 입력 수를 줄여줍니다. 비싼 LLMChainExtractor를 마지막에 두면 EmbeddingsFilter가 이미 걸러낸 소수의 문서에만 LLM을 적용하게 되어 비용을 크게 줄일 수 있어요.

> 💡 **실무 팁**: 파이프라인 단계 순서가 중요합니다. 비용이 낮은 단계(TextSplitter, EmbeddingsFilter)를 먼저 적용해 문서를 줄이고, 비용이 높은 단계(LLMChainExtractor)는 마지막에 적용하세요.

In [28]:
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

# 1. 텍스트 분할기 (문서를 더 작은 청크로)
splitter = CharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    # separator="<br/>"
)

# 2. 중복 필터 (유사도 0.95 이상인 문서 제거)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

# 3. 관련성 필터 (유사도 0.75 이상만 유지)
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.472
)

# 4. LLM 내용 추출기
extractor = LLMChainExtractor.from_llm(llm)

# 파이프라인 생성
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter, extractor]
)

print("✅ DocumentCompressorPipeline 생성 완료")
print("<br/>파이프라인 단계:")
print("  1. 텍스트 분할 (400자 청크)")
print("  2. 중복 문서 제거")
print("  3. 관련성 필터링 (유사도 ≥ 0.472)")
print("  4. LLM 내용 추출")


✅ DocumentCompressorPipeline 생성 완료
<br/>파이프라인 단계:
  1. 텍스트 분할 (400자 청크)
  2. 중복 문서 제거
  3. 관련성 필터링 (유사도 ≥ 0.472)
  4. LLM 내용 추출


In [29]:
# ---------------------------------------------------
# 파이프라인 적용 후 모든 방식 비교
# ---------------------------------------------------

# ============================================================
# TODO: pipeline_retriever로 검색하고 4가지 방식의 문서 수와 총 문자 수를 비교하세요
# 힌트: pipeline_retriever.invoke(query) 후 비교표 출력
# 예상 결과: Pipeline이 가장 압축된 결과를 반환
# ============================================================

# 파이프라인 적용
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

# 파이프라인 실행
pipeline_docs = pipeline_retriever.invoke(query)

print(f"📝 쿼리: {query}<br/>")
print_docs(pipeline_docs, "DocumentCompressorPipeline 적용 후")

# 전체 비교
print(f"<br/>📊 압축 효과 비교:")
print(f"{'방식':<30} {'문서수':<10} {'총 문자수':<15}")
print("-" * 60)
print(f"{'기본 Retriever':<30} {len(docs):<10} {sum(len(d.page_content) for d in docs):<15}")
print(f"{'LLMChainExtractor':<30} {len(compressed_docs):<10} {sum(len(d.page_content) for d in compressed_docs):<15}")
print(f"{'EmbeddingsFilter':<30} {len(embedding_filtered_docs):<10} {sum(len(d.page_content) for d in embedding_filtered_docs):<15}")
print(f"{'Pipeline (all combined)':<30} {len(pipeline_docs):<10} {sum(len(d.page_content) for d in pipeline_docs):<15}")

📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?<br/>

DocumentCompressorPipeline 적용 후

[문서 1]
내용: "Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으로써 발생하는 병렬 처리의 이점을 강조하며, 이를 통해 훨씬 더 빠른 학습 속도와 효율적인 메모리 사용을 실현합니다.
길이: 265 문자
--------------------------------------------------------------------------------

[문서 2]
내용: "Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Transformer) 모델의 아키텍처를 처음으로 소개한 연구입니다. 변환기 모델은 이전의 순환 신경망(RNN)이나 합성곱 신경망(CNN)에 의존하던 방식에서 벗어나, 전적으로 '어텐션 메커니즘(Attention Mechanism)'을 사용하여 정보를 처리합니다. 이로 인해 모델의 학습 속도와 효율성이 크게 향상되었습니다.
길이: 231 문자
--------------------------------------------------------------------------------

<br/>📊 압축 효과 비교:
방식                             문서수        총 문자수          
------------------------------------------------------------
기본 Retriever                   3          2107           
LLMChainExtractor     

In [30]:
# ---------------------------------------------------
# 파이프라인 각 단계별 중간 결과 관찰
# ---------------------------------------------------
from langchain_core.documents.compressor import BaseDocumentCompressor
from langchain_core.documents.transformers import BaseDocumentTransformer

stage_names = [
    "1단계: CharacterTextSplitter (400자 청크 분할)",
    "2단계: EmbeddingsRedundantFilter (중복 제거)",
    "3단계: EmbeddingsFilter (유사도 ≥ 0.472)",
    "4단계: LLMChainExtractor (LLM 내용 추출)",
]

# Base Retriever 결과를 시작점으로 사용
step_docs = base_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print(f"{'='*70}")
print(f"[입력] Base Retriever")
print(f"{'='*70}")
print(f"  문서 수: {len(step_docs)}개 | 총 문자 수: {sum(len(d.page_content) for d in step_docs):,}자")
for i, doc in enumerate(step_docs, 1):
    print(f"  문서 {i}: {len(doc.page_content):>4}자 | {doc.page_content[:50]}...")

# 파이프라인의 각 transformer를 순차 적용
for idx, transformer in enumerate(pipeline_compressor.transformers):
    if isinstance(transformer, BaseDocumentCompressor):
        step_docs = list(transformer.compress_documents(step_docs, query))
    elif isinstance(transformer, BaseDocumentTransformer):
        step_docs = list(transformer.transform_documents(step_docs))

    total_chars = sum(len(d.page_content) for d in step_docs)

    print(f"\n{'='*70}")
    print(f"[{stage_names[idx]}]")
    print(f"{'='*70}")
    print(f"  문서 수: {len(step_docs)}개 | 총 문자 수: {total_chars:,}자")
    for i, doc in enumerate(step_docs, 1):
        print(f"  문서 {i}: {len(doc.page_content):>4}자 | {doc.page_content[:50]}...")

# 요약 테이블
print(f"\n{'='*70}")
print(f"📊 단계별 변화 요약")
print(f"{'='*70}")
print(f"{'단계':<45} {'문서수':>6} {'문자수':>8}")
print(f"{'-'*62}")

# 다시 실행하여 수치 수집
track = base_retriever.invoke(query)
print(f"{'[입력] Base Retriever':<45} {len(track):>6} {sum(len(d.page_content) for d in track):>8,}")
for idx, transformer in enumerate(pipeline_compressor.transformers):
    if isinstance(transformer, BaseDocumentCompressor):
        track = list(transformer.compress_documents(track, query))
    elif isinstance(transformer, BaseDocumentTransformer):
        track = list(transformer.transform_documents(track))
    print(f"{stage_names[idx]:<45} {len(track):>6} {sum(len(d.page_content) for d in track):>8,}")

📝 쿼리: 딥러닝에서 Attention 메커니즘은 어떻게 작동하나요?

[입력] Base Retriever
  문서 수: 3개 | 총 문자 수: 2,107자
  문서 1:  721자 | 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울...
  문서 2:  681자 | Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과...
  문서 3:  705자 | 이 논문은 기계 학습 및 NLP 분야에서 기존의 접근 방식을 재고하고, 어텐션 메커니즘의 ...

[1단계: CharacterTextSplitter (400자 청크 분할)]
  문서 수: 8개 | 총 문자 수: 2,146자
  문서 1:  249자 | 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울...
  문서 2:  265자 | "Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '...
  문서 3:  203자 | 변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔...
  문서 4:  371자 | Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과...
  문서 5:  335자 | Attention is all you need

"Attention Is All You N...
  문서 6:  174자 | 이 논문은 기계 학습 및 NLP 분야에서 기존의 접근 방식을 재고하고, 어텐션 메커니즘의 ...
  문서 7:  250자 | ARIMA (자기회귀 누적 이동평균), SARIMA (계절성 자기회귀 누적 이동평균), 그...
  문서 8:  299자 | ARIMA (자기회귀 누적 이동평균)

ARIMA 모델은 비계절적 시계열 데이터를 분석하고...

[2단계: EmbeddingsRedundantFilter (중복 제거)]
  문서 수: 8개 | 총 문자 수: 2,146자
 

```mermaid
graph TD
    subgraph pipeline ["DocumentCompressorPipeline 처리 흐름"]
        direction TB
        A["Base Retriever<br/>원본 문서 3개<br/>2,107자"] -->|"문서 분할"| B["CharacterTextSplitter<br/>chunk_size=400<br/>작은 청크로 분할"]
        B -->|"중복 제거"| C["EmbeddingsRedundantFilter<br/>유사도 0.95 이상 제거"]
        C -->|"관련성 필터링"| D["EmbeddingsFilter<br/>similarity ≥ 0.472"]
        D -->|"내용 추출"| E["LLMChainExtractor<br/>LLM이 핵심만 추출"]
        E --> F["최종 결과<br/>2개 문서, 496자"]
    end

    subgraph compare ["방식별 비교"]
        direction LR
        G["기본 Retriever<br/>3문서 · 2,107자"] --> H["LLMChainExtractor<br/>2문서 · 727자<br/>감소율 65%"]
        G --> I["EmbeddingsFilter<br/>1문서 · 721자<br/>감소율 66%"]
        G --> J["Pipeline<br/>2문서 · 496자<br/>감소율 76%"]
    end

    style A fill:#e3f2fd,stroke:#1976d2,color:#0d47a1
    style B fill:#fff3e0,stroke:#f57c00,color:#e65100
    style C fill:#fff3e0,stroke:#f57c00,color:#e65100
    style D fill:#fff3e0,stroke:#f57c00,color:#e65100
    style E fill:#fff3e0,stroke:#f57c00,color:#e65100
    style F fill:#e8f5e9,stroke:#388e3c,color:#1b5e20
    style G fill:#f3e5f5,stroke:#7b1fa2,color:#4a148c
    style H fill:#fce4ec,stroke:#c62828,color:#b71c1c
    style I fill:#fce4ec,stroke:#c62828,color:#b71c1c
    style J fill:#e8f5e9,stroke:#388e3c,color:#1b5e20
```

> **핵심 포인트**: Pipeline은 문서를 먼저 **작은 청크로 분할**한 뒤 필터링하므로, EmbeddingsFilter 단독(1문서)보다 문서 **수**는 늘어날 수 있지만 총 **문자 수**(496자)는 가장 적어요. 여러 원본 문서에서 관련 부분만 정밀하게 추출하는 것이 Pipeline의 장점입니다.

## 6. Compressor 선택 가이드

### 사용 시나리오별 추천

| 시나리오 | 추천 Compressor | 이유 |
|---------|----------------|------|
| **최고 품질 필요** | LLMChainExtractor | 가장 정확한 내용 추출 |
| **빠른 응답 필요** | EmbeddingsFilter | LLM 호출 없음 |
| **비용 최소화** | EmbeddingsFilter | 임베딩만 사용 |
| **문서 선택만** | LLMChainFilter | 원본 유지하며 필터링 |
| **최적 균형** | Pipeline | 단계별 최적화 |

### 비용 vs 품질 트레이드오프

```
높은 품질 ← → 낮은 비용/빠른 속도

LLMChainExtractor > Pipeline > LLMChainFilter > EmbeddingsFilter
```

### 실전 권장 조합

```python
# Production 환경 권장
pipeline = DocumentCompressorPipeline([
    CharacterTextSplitter(chunk_size=400),  # 작은 청크
    EmbeddingsRedundantFilter(),           # 중복 제거
    EmbeddingsFilter(threshold=0.75),      # 빠른 필터링
    # LLMChainExtractor()  # 선택적: 최종 품질 향상
])
```


## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| Compressor | 비용 | 속도 | 품질 | 추천 상황 |
|-----------|------|------|------|----------|
| **LLMChainExtractor** | 높음 | 느림 | 최고 | 품질이 최우선 |
| **LLMChainFilter** | 높음 | 느림 | 높음 | 원본 유지하며 필터링 |
| **EmbeddingsFilter** | 낮음 | 빠름 | 보통 | 비용·속도 중시 |
| **Pipeline** | 중간 | 중간 | 높음 | 균형 잡힌 운영 |

### 다음 노트북 예고

**03-EnsembleRetriever**: BM25 키워드 검색과 벡터 검색을 결합하는 하이브리드 검색을 배워요. 검색 방식의 약점을 서로 보완하는 실무 필수 기법입니다.